# Continuum — Interactive Recovery Demo

Drives the kill-and-recover sequence against a running Continuum API and shows the
CockroachDB memory layer doing the work — **without cloning or installing anything**
if you point `BASE_URL` at a running instance.

> The claim under test: *the agent's execution environment is allowed to die mid-incident;
> its memory is not.*

**What you need:** a reachable Continuum API. Either
- local — `make run-api` (then `BASE_URL = "http://localhost:8000"`), or
- your own deployment.

The kill beat (sections 4–5) requires a **local** API, because it kills the process by port.
Against a remote URL, run sections 1–3 only.

Recording script for the same flow: [`submission/DEMO_SCRIPT.md`](../submission/DEMO_SCRIPT.md).
Setup notes: [`notebooks/README.md`](README.md).

In [ ]:
import json

import httpx

BASE_URL = "http://localhost:8000"  # or your deployment


def show(obj):
    print(json.dumps(obj, indent=2, default=str))


r = httpx.get(f"{BASE_URL}/api/v1/health", timeout=10)
r.raise_for_status()
show(r.json())

## 1. Fire a synthetic alert

All incident data is synthetic (ADR 005). The orchestrator's **first** action on every
invocation is a CockroachDB read for existing open incident state — never a warm cache.

In [ ]:
alert = {
    "service": "checkout-api",
    "severity": "high",
    "text": "Connection pool exhausted; p99 latency 4200ms; 503s rising on /checkout",
}

r = httpx.post(f"{BASE_URL}/api/v1/alert", json=alert, timeout=60)
r.raise_for_status()
result = r.json()
show(result)

## 2. Read the live state back over MCP

`/api/v1/incidents/open` is answered by the **application** calling the CockroachDB Cloud
Managed MCP Server's read-only SQL tool — not by a developer in an IDE (ADR 003).

A `503` here is a legitimate, deliberate response: the endpoint surfaces MCP failure rather
than masking it.

In [ ]:
r = httpx.get(f"{BASE_URL}/api/v1/incidents/open", timeout=30)
print(f"HTTP {r.status_code}")
show(r.json())

## 3. Advance the incident one step

Each step commits in **two** explicit `SERIALIZABLE` transactions — `executing` before the
execution window, `executed` after (ADR 009). The gap between them is where a crash lands.

In [ ]:
r = httpx.post(f"{BASE_URL}/api/v1/alert", json=alert, timeout=60)
show(r.json())

## 4. The kill — local only

**This is the point of the project.** A real `SIGKILL` / `TerminateProcess`: no graceful
shutdown, no checkpoint call, no cleanup hook. Fire it *during* a step's execution window
so the step is durably stuck in `executing`.

Run this from a shell next to a live `make run-api`:

```bash
python scripts/demo_run.py --tick --via-api   # start a step (in one terminal)
python scripts/chaos_kill.py --port 8000      # kill it mid-step (in another)
```

`--via-api` is required: a bare `--tick` runs in-process and finishes before you could kill it.

## 5. Confirm the state outlived the process

With the API dead, query CockroachDB directly. A row sitting in `executing` with no process
alive to own it **is** the thesis — this is the shot the demo video leads with.

Needs `COCKROACH_DATABASE_URL` in the environment.

In [ ]:
import os

import psycopg

with psycopg.connect(os.environ["COCKROACH_DATABASE_URL"]) as conn:
    rows = conn.execute(
        """
        SELECT incident_id, step_index, action, state, started_at, completed_at
        FROM remediation_steps
        ORDER BY started_at DESC
        LIMIT 10
        """
    ).fetchall()

for row in rows:
    marker = "  <-- died here" if row[3] == "executing" else ""
    print(f"step {row[1]}  {row[3]:<10} {row[2]}{marker}")

## 6. The recovery

Restart the API and tick again — or, with the orchestrator deployed, let a genuinely cold
Lambda invocation do it:

```bash
make run-api                                        # cold restart
python scripts/demo_run.py --tick --via-api --resume-check
# or, deployed:
python scripts/demo_run.py --tick --via-lambda
```

The interrupted step is **re-executed, not skipped and not duplicated** — then committed
`executed` before the next step is claimed. Re-run cell 5 to watch the frozen row advance.

The exactly-once property under concurrent invocations is asserted against a real cluster in
[`tests/integration/test_recovery_e2e.py`](../tests/integration/test_recovery_e2e.py), and the
literal process-kill in
[`tests/integration/test_chaos_kill_e2e.py`](../tests/integration/test_chaos_kill_e2e.py).